In [7]:
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [8]:
from google.colab import files
uploaded = files.upload()


Saving formatted_movie_lines.txt to formatted_movie_lines (1).txt


In [12]:
# Load file
with open("formatted_movie_lines.txt", encoding="utf-8") as f:
    lines = f.readlines()

# Use only 25000 lines (IMPORTANT)
lines = lines[:25000]

# Clean text
corpus = [line.strip().lower() for line in lines if len(line.strip()) > 5]

In [13]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(corpus)

total_words = 5000

In [14]:
input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

In [15]:
max_sequence_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

input_sequences = np.array(input_sequences)

In [16]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [17]:
model = Sequential()

model.add(Input(shape=(max_sequence_len-1,)))
model.add(Embedding(5000, 100))

model.add(LSTM(120))

model.add(Dense(5000, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 528, 100)       │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 120)            │       106,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5000)           │       605,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,211,080 (4.62 MB)

 Trainable params: 1,211,080 (4.62 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.fit(X, y, epochs=25, batch_size=64)

Epoch 1/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 218s 27ms/step - accuracy: 0.1091 - loss: 5.5528
Epoch 2/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 212s 27ms/step - accuracy: 0.1454 - loss: 5.0145
Epoch 3/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 217s 28ms/step - accuracy: 0.1585 - loss: 4.7840
Epoch 4/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 263s 28ms/step - accuracy: 0.1683 - loss: 4.6138
Epoch 5/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 217s 28ms/step - accuracy: 0.1770 - loss: 4.4714
Epoch 6/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 218s 28ms/step - accuracy: 0.1862 - loss: 4.3461
Epoch 7/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 219s 28ms/step - accuracy: 0.1956 - loss: 4.2335
Epoch 8/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 219s 28ms/step - accuracy: 0.2047 - loss: 4.1318
Epoch 9/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 220s 28ms/step - accuracy: 0.2145 - loss: 4.0400
Epoch 10/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 220s 28ms/step - accuracy: 0.2242 - loss: 3.9570
Epoch 11/25
7735/7735 ━━━━━━━━━━━━━━━━━━━━ 220s 28ms/step - accuracy: 0.2328 - loss: 3.88

In [19]:
model.save("next_word_model.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [20]:
def predict_next_words(text, next_words=5):
    text = text.lower()

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]

        word = tokenizer.index_word.get(predicted, "")

        if word == "" or word == "<OOV>":
            break

        text += " " + word

    return text

In [21]:
model.save("next_word_model.h5")

In [22]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [23]:
from google.colab import files

files.download("next_word_model.h5")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
loss, accuracy = model.evaluate(X, y, verbose=1)

print("Accuracy:", accuracy)
print("Loss:", loss)

15470/15470 ━━━━━━━━━━━━━━━━━━━━ 187s 12ms/step - accuracy: 0.3391 - loss: 3.1564
Accuracy: 0.3390864431858063
Loss: 3.1564085483551025
